# Prueba SQL

El coronavirus tomó al mundo entero por sorpresa, cambiando la rutina diaria de todos y todas. Los habitantes de las ciudades ya no pasaban su tiempo libre fuera, yendo a cafés y centros comerciales; sino que más gente se quedaba en casa, leyendo libros. Eso atrajo la atención de las startups (empresas emergentes) que se apresuraron a desarrollar nuevas aplicaciones para los amantes de los libros.

Te han dado una base de datos de uno de los servicios que compiten en este mercado. Contiene datos sobre libros, editoriales, autores y calificaciones de clientes y reseñas de libros. Esta información se utilizará para generar una propuesta de valor para un nuevo producto.

## 1. Importar librerias

In [4]:
# importar librerías
import pandas as pd
from sqlalchemy import create_engine
import datetime as dt
import matplotlib.pyplot as plt



db_config = {'user': 'practicum_student',         # nombre de usuario
             'pwd': 's65BlTKV3faNIGhmvJVzOqhs', # contraseña
             'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
             'port': 6432,              # puerto de conexión
             'db': 'data-analyst-final-project-db'}          # nombre de la base de datos

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
                                                                     db_config['pwd'],
                                                                       db_config['host'],
                                                                       db_config['port'],
                                                                       db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

## 2. Exploracion de datos

In [5]:
# Crear una consulta SQL.
query = ''' SELECT *
            FROM books
        '''

In [6]:
books = pd.io.sql.read_sql(query, con = engine)
books.head()

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


In [7]:
# Crear una consulta SQL.
query1 = ''' SELECT *
            FROM authors
        '''

In [8]:
authors = pd.io.sql.read_sql(query1, con = engine)
authors.head()

,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


In [9]:
# Crear una consulta SQL.
query2 = ''' SELECT *
            FROM publishers
        '''

In [10]:
publishers = pd.io.sql.read_sql(query2, con = engine)
publishers.head()

,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


In [11]:
# Crear una consulta SQL.
query3 = ''' SELECT *
            FROM ratings
        '''

In [12]:
ratings = pd.io.sql.read_sql(query3, con = engine)
ratings.head()

,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


In [13]:
# Crear una consulta SQL.
query4 = ''' SELECT *
            FROM reviews
        '''

In [14]:
reviews = pd.io.sql.read_sql(query4, con = engine)
reviews.head()

,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


## 3 Analisis y extracion de la información

### 3.1 Analisis de libros publicados

¿Cuantos libros fueron publicados despues del 1 de enero del 2000?

In [15]:
books_before_date =   ''' SELECT COUNT(title)
            FROM books
            WHERE publication_date > '2000/01/01'
        '''
pd.io.sql.read_sql(books_before_date, con = engine)

,count
0,819


Despues del 1 de enero del 2000 se publicaron 819 de libros.

### 3.2 Analisis de reseñas y calificaciones por libro

¿Cual es el numero de reseñas y la calificacion promedio para cada libro?

Para poder hallar las reseñas y las calificaciones uniremos las tablas raitings y reviews junto a la tabla books

In [16]:
rating_bbok = ''' SELECT book_id, AVG(rating) as avg_rating
        FROM ratings
        GROUP BY book_id
        ORDER BY book_id
        LIMIT 5;
        
        '''
#agrupamos los book_id y promediamos el rating

pd.io.sql.read_sql(rating_bbok, con = engine)

,book_id,avg_rating
0,1,3.666667
1,2,2.500000
2,3,4.666667
3,4,4.500000
4,5,4.000000


In [17]:
review_bbok = ''' SELECT book_id, COUNT(text) AS count_reviews
        FROM reviews
        GROUP BY book_id
        ORDER BY book_id
        LIMIT 5;
            
        '''

#agrupamos por book_id y contamos las reseñas
pd.io.sql.read_sql(review_bbok, con = engine)

,book_id,count_reviews
0,1,2
1,2,1
2,3,3
3,4,2
4,5,4


In [18]:
join_tables = ''' SELECT 
avg_table.book_id,
avg_table.avg_rating,
count_table.count_reviews

FROM (

    SELECT book_id, AVG(rating) as avg_rating
        FROM ratings
        GROUP BY book_id
        ORDER BY book_id
) AS avg_table

JOIN (

    SELECT book_id, COUNT(text) AS count_reviews
        FROM reviews
        GROUP BY book_id
        ORDER BY book_id
 
) AS count_table

ON avg_table.book_id = count_table.book_id

''' 

#unimos las tablas anteriores del promedio de rating y el conteo de reviews

pd.io.sql.read_sql(join_tables, con = engine)

,book_id,avg_rating,count_reviews
0,1,3.666667,2
1,2,2.500000,1
2,3,4.666667,3
3,4,4.500000,2
4,5,4.000000,4
...,...,...,...
989,996,3.666667,3
990,997,3.400000,3
991,998,3.200000,4
992,999,4.500000,2


Con estas agrupaciones ya tenemos un panorama mejor para poder unir las tablas. 

In [19]:
main_table = ''' SELECT 
main_table.book_id,
main_table.title,
temp_table.avg_rating,
temp_table.count_reviews

FROM (

    SELECT book_id, title
    FROM books

) AS main_table

JOIN (
 
    SELECT 
avg_table.book_id,
avg_table.avg_rating,
count_table.count_reviews

FROM (

    SELECT book_id, AVG(rating) as avg_rating
        FROM ratings
        GROUP BY book_id
        ORDER BY book_id
) AS avg_table

JOIN (

    SELECT book_id, COUNT(text) AS count_reviews
        FROM reviews
        GROUP BY book_id
        ORDER BY book_id
 
) AS count_table

ON avg_table.book_id = count_table.book_id
) AS temp_table

ON main_table.book_id = temp_table.book_id
ORDER  BY count_reviews DESC
LIMIT 5
''' 
#unimos por la columna book_id con la tabla anterior (rating y reviews) con la tabla books
pd.io.sql.read_sql(main_table, con = engine)

,book_id,title,avg_rating,count_reviews
0,948,Twilight (Twilight #1),3.662500,7
1,497,Outlander (Outlander #1),4.125000,6
2,207,Eat Pray Love,3.395833,6
3,299,Harry Potter and the Chamber of Secrets (Harry...,4.287500,6
4,302,Harry Potter and the Prisoner of Azkaban (Harr...,4.414634,6


### 3.3 Analisis de editoriales

¿Qué editorial ha publicado mas libros con mas de 50 paginas?

In [20]:
publishers_books = '''SELECT 
books.book_id, 
books.title,
books.publisher_id,
books.num_pages,
publishers.publisher_id,
publishers.publisher

FROM books
JOIN publishers

ON books.publisher_id = publishers.publisher_id

WHERE num_pages > 50

'''
#unimos la tabbla join con la tabla publishers

pd.io.sql.read_sql(publishers_books, con = engine)

,book_id,title,publisher_id,num_pages,publisher_id,publisher
0,1,'Salem's Lot,93,594,93,Doubleday
1,2,1 000 Places to See Before You Die,336,992,336,Workman Publishing Company
2,3,13 Little Blue Envelopes (Little Blue Envelope...,135,322,135,HarperCollins Publishers
3,4,1491: New Revelations of the Americas Before C...,309,541,309,Vintage
4,5,1776,268,386,268,Simon Schuster
...,...,...,...,...,...,...
987,996,Wyrd Sisters (Discworld #6; Witches #2),147,265,147,Hartorch
988,997,Xenocide (Ender's Saga #3),297,592,297,Tor Books
989,998,Year of Wonders,212,358,212,Penguin Books
990,999,You Suck (A Love Story #2),331,328,331,William Morrow


In [21]:
count_books = '''SELECT 
    publishers.publisher,
    COUNT(books.book_id) AS num_books
FROM books
JOIN publishers
    ON books.publisher_id = publishers.publisher_id
WHERE books.num_pages > 50
GROUP BY publishers.publisher
ORDER BY num_books DESC
LIMIT 5;


'''
#agrupamos por editorial para revisar quienes publicaron mas libros y mostramos las 5 primeras editoriales
pd.io.sql.read_sql(count_books, con = engine)

,publisher,num_books
0,Penguin Books,42
1,Vintage,31
2,Grand Central Publishing,25
3,Penguin Classics,24
4,Bantam,19


### 3.4 Analisis de autores con mejor calificacion promedio

¿Quien es el autor con una mejor calificacion promedio en libros con al menos 50 calificaciones?

In [22]:
author_books = '''SELECT 
books.book_id,
books.author_id,
books.title,
books.num_pages,
books.publication_date,
books.publisher_id,
authors.author

FROM books 
JOIN authors
ON books.author_id = authors.author_id
LIMIT 5
'''
pd.io.sql.read_sql(author_books, con = engine)

,book_id,author_id,title,num_pages,publication_date,publisher_id,author
0,1,546,'Salem's Lot,594,2005-11-01,93,Stephen King/Jerry N. Uelsmann
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336,Patricia Schultz
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135,Maureen Johnson
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309,Charles C. Mann
4,5,125,1776,386,2006-07-04,268,David McCullough


In [23]:
ratings.head()

,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


In [24]:
rating_authors = ''' WITH BookRatings AS (
SELECT
    books_r.book_id,
    books_r.author_id,
    books_r.author,
    AVG(ratings.rating) AS avg_rating,
    COUNT(ratings.rating) AS num_rating
FROM (

    SELECT 
        books.book_id,
        books.author_id,
        books.title,
        books.num_pages,
        books.publication_date,
        books.publisher_id,
        authors.author
    FROM books 
    JOIN authors
    ON books.author_id = authors.author_id
) AS books_r

JOIN ratings
ON books_r.book_id = ratings.book_id
GROUP BY books_r.author_id, books_r.author, books_r.book_id
HAVING COUNT(ratings.rating) >= 50 
)

SELECT
    author,
    AVG(avg_rating) AS author_avg_rating
FROM BookRatings
GROUP BY author
ORDER BY author_avg_rating DESC
LIMIT 1;
'''

pd.io.sql.read_sql(rating_authors, con = engine)

,author,author_avg_rating
0,J.K. Rowling/Mary GrandPré,4.283844


El autor con una mejor calificacion promedio en libros con al menos 50 calificaciones es J.K Rowling/Mary GrandPré.

### 3.5 Analisis de reseñas

¿Cuantas reseñas de texto en promedion hacen los usuarios?

In [25]:
reviews.head()

,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


In [26]:
merge_reviews = ''' WITH AvgReviews AS (SELECT 
    reviews_avg.username,
    reviews_avg.count_text,
    rating_avg.count_books
FROM (
    SELECT 
        username, 
        COUNT(text) AS count_text
    FROM reviews
    GROUP BY username
    ORDER BY count_text DESC
) AS reviews_avg
RIGHT JOIN (
    SELECT 
        username, 
        COUNT(book_id) AS count_books
    FROM ratings
    GROUP BY username
    HAVING COUNT(book_id) > 50
    ORDER BY count_books DESC
) AS rating_avg
ON reviews_avg.username = rating_avg.username

)

SELECT
    AVG(count_text) as avg_count_text
    
FROM AvgReviews
'''


pd.io.sql.read_sql(merge_reviews, con = engine)

,avg_count_text
0,24.333333


Los usuarios hace en promedio 24 reseñas de texto.

## 4. Conclusiones

* Desde el 1 de enero del 2000, se han lanzado 819 libros, lo que demuestra una actividad editorial continua y un amplio abanico de opciones para los lectores de hoy en día. Entre las editoriales más destacadas se encuentran Penguin Books, Vintage y Grand Central Publishing, siendo Penguin la que ha publicado más títulos (42), lo que refuerza su liderazgo en la industria literaria.

* En cuanto a los autores, J.K. Rowling/Mary GrandPré ocupa el primer lugar en calificaciones promedio entre aquellos con al menos 50 reseñas, lo que resalta su impacto en la literatura contemporánea, especialmente en el género de fantasía.

* Los usuarios participan activamente en la comunidad, dejando un promedio de 24 reseñas por persona, lo que refleja un fuerte interés por compartir opiniones y ayudar a otros lectores a decidir qué libros elegir. Entre los títulos mejor calificados, destacan A Woman of Substance y Act of Treason con un 5.0 de promedio, aunque su bajo número de reseñas podría influir en estos resultados.

* Finalmente, en cuanto a los libros con más reseñas, títulos como Twilight, Outlander y las sagas de Harry Potter siguen siendo muy populares, aunque sus calificaciones varían. El alto volumen de reseñas sugiere que estos libros han capturado la atención de una amplia audiencia.